# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trycatchqasim/ML_FR_Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window



Unit of Analysis: One row represents a single content item for a specific client on a single day (report_date $\times$ client_id $\times$ content_id).  

Table(s) Used: fact_content_daily_performance joined with dim_content for content-level attributes.

  Time Window: Mid-panel month month=2026-03 (2026-03-01 to 2026-03-31) to prevent practicing on the final evaluation month (June 2026).
  
   Target / Label / Proxy: Binary classification of whether a content item experiences an engagement drop or is marked declining (is_declining_label).

   
  Deliberately Excluded: trend_direction and trend_pct, because the decline label is directly computed from trend_pct (creating 100% target leakage).


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [21]:
import duckdb
import pandas as pd
import numpy as np
from huggingface_hub import HfFileSystem, login
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc
from google.colab import userdata

# 1. Login to Hugging Face
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# 2. Discover the exact parquet file list via HfFileSystem
fs = HfFileSystem(token=hf_token)
repo_path = "datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

all_files = fs.glob(f"{repo_path}/**/*.parquet")
month_files = [f"hf://{f}" for f in all_files if "month=2026-03" in f]

# If exact month folder is named without prefix, grab the March partition
if not month_files:
    month_files = [f"hf://{f}" for f in all_files if "2026-03" in f]

# Direct URL array string for DuckDB read_parquet
file_list_str = str(month_files)
print(f"Found {len(month_files)} parquet file(s) for 2026-03.")

# 3. Setup DuckDB with HF Secret
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HTTP,
        EXTRA_HTTP_HEADERS MAP {{'Authorization': 'Bearer {hf_token}'}}
    );
""")

Found 1 parquet file(s) for 2026-03.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt
FROM read_parquet({file_list_str})
GROUP BY report_date, client_hash_id, content_hash_id
HAVING cnt > 1
LIMIT 5;
"""

grain_df = con.execute(grain_query).df()
print(f"Grain violations found: {len(grain_df)}")
display(grain_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations found: 0


,report_date,client_hash_id,content_hash_id,cnt


In [23]:
span_query = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count
FROM read_parquet({file_list_str});
"""

span_df = con.execute(span_query).df()
display(span_df)

,total_rows,min_date,max_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


In [24]:
avail_query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) AS ga4_surviving_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS gsc_surviving_rows,
    COUNT(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 END) AS both_active_rows,
    ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct
FROM read_parquet({file_list_str});
"""

avail_df = con.execute(avail_query).df()
display(avail_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_surviving_rows,gsc_surviving_rows,both_active_rows,survival_pct
0,9841378,413966,3611061,364347,3.7


### 3) Five Features & Timing Justifications

1. **`gsc_impressions`**: Knowable at decision time because it reflects raw search visibility recorded on or before `report_date`.
2. **`gsc_clicks`**: Knowable at decision time because it measures logged user clicks up to the current daily batch.
3. **`gsc_avg_position`**: Knowable at decision time because it captures search ranking history prior to the forward prediction window (with `0` tracked as unranked).
4. **`ga4_sessions`**: Knowable at decision time because it records onsite sessions accumulated prior to inference.
5. **`has_position_flag`**: Knowable at decision time as a binary indicator (`gsc_avg_position > 0`) preventing category-imputation signal leakage.

In [27]:
# 1. Query daily performance features with an explicit target and a deliberate future-leak trap
extract_query = f"""
WITH daily_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        -- Honest Features (Known on report_date)
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_clicks, 0) AS gsc_clicks,
        COALESCE(gsc_avg_position, 0) AS gsc_avg_position,
        COALESCE(ga4_sessions, 0) AS ga4_sessions,
        CASE WHEN gsc_avg_position > 0 THEN 1 ELSE 0 END AS has_position_flag,

        -- Target: Binary indicator if content has high search engagement (e.g., gsc_clicks >= 5)
        CASE WHEN COALESCE(gsc_clicks, 0) >= 5 THEN 1 ELSE 0 END AS target,

        -- Deliberate Leakage Trap: A feature derived directly from the target signal
        -- (e.g., exact click count or an outcome proxy knowable only when target is known)
        COALESCE(gsc_clicks, 0) * 1.0 AS leaked_signal
    FROM read_parquet({file_list_str})
    WHERE ga4_data_available IS TRUE
      AND gsc_data_available IS TRUE
    LIMIT 50000
)
SELECT * FROM daily_data;
"""

df = con.execute(extract_query).df()

# 2. Define honest vs leaked feature sets (excluding raw gsc_clicks from honest set if it defines the target)
honest_cols = ['gsc_impressions', 'gsc_avg_position', 'ga4_sessions', 'has_position_flag']
leaked_cols = honest_cols + ['leaked_signal']

X_train, X_test, y_train, y_test = train_test_split(df, df['target'], test_size=0.3, random_state=42)

# --- Fit Honest Model ---
rf_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_honest.fit(X_train[honest_cols], y_train)
probs_honest = rf_honest.predict_proba(X_test[honest_cols])[:, 1]
precision_h, recall_h, _ = precision_recall_curve(y_test, probs_honest)
honest_pr_auc = auc(recall_h, precision_h)

# --- Fit Leaked Model (The Trap) ---
rf_leak = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_leak.fit(X_train[leaked_cols], y_train)
probs_leak = rf_leak.predict_proba(X_test[leaked_cols])[:, 1]
precision_l, recall_l, _ = precision_recall_curve(y_test, probs_leak)
leaked_pr_auc = auc(recall_l, precision_l)

print(f"--- Leakage Experiment Results ---")
print(f"Honest Model PR-AUC (clean features): {honest_pr_auc:.4f}")
print(f"Leaked Model PR-AUC (with leaked trap): {leaked_pr_auc:.4f}")

# --- Purge Leakage ---
del df['leaked_signal']
print("\n[SUCCESS] Leaked column purged. Honest baseline metric retained.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Leakage Experiment Results ---
Honest Model PR-AUC (clean features): 0.7046
Leaked Model PR-AUC (with leaked trap): 1.0000

[SUCCESS] Leaked column purged. Honest baseline metric retained.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


Slice Limitation

Limitation: Filtering on ga4_data_available IS TRUE drops uninstrumented early-history client content and can bias the dataset toward established clients with complete tracking setups.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.